# Course 8: Privacy-Preserving Machine Learning
## Session 3: DP-SGD and privacy accounting

**Goal:** train a classifier with DP-SGD written by hand, compute epsilon with a Renyi accountant, and plot accuracy against epsilon.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. DP-SGD in 25 lines, and an accountant
Three changes to ordinary SGD: sample each example with probability q, clip every example's gradient to length `clip`, and add Gaussian noise of scale `sigma * clip` to the sum.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from math import comb, exp, log, sqrt
from torch.func import functional_call, vmap, grad

# ---- privacy accounting (Renyi DP of the sampled Gaussian mechanism, whole-number orders) ----
def rdp(q, sigma, alpha):
    """Renyi DP at order alpha of one DP-SGD step. q = sampling rate, sigma = noise multiplier."""
    if q == 1.0: return alpha / (2 * sigma ** 2)
    terms = [log(comb(alpha, k)) + (alpha - k) * log(1 - q) + k * log(q) + (k * k - k) / (2 * sigma ** 2) for k in range(alpha + 1)]
    m = max(terms)
    return (m + log(sum(exp(t - m) for t in terms))) / (alpha - 1)

def epsilon(q, sigma, steps, delta):
    """Convert composed Renyi DP to (epsilon, delta)-DP and keep the best order."""
    return min(steps * rdp(q, sigma, a) + log(1 / delta) / (a - 1) for a in range(2, 129))

# ---- DP-SGD ----
def dp_sgd(model, X, y, sigma, clip=1.0, lr=0.5, steps=300, q=0.1, seed=0, loss_fn=F.cross_entropy):
    """Poisson sampling, per-example clipping, Gaussian noise. sigma=0 with clip=inf is ordinary SGD."""
    g = torch.Generator().manual_seed(seed)
    params = {k: v.detach().clone() for k, v in model.named_parameters()}
    one = lambda p, x, t: loss_fn(functional_call(model, p, (x[None],)), t[None])
    per_example_grads = vmap(grad(one), in_dims=(None, 0, 0))
    n = len(X)
    for _ in range(steps):
        idx = torch.nonzero(torch.rand(n, generator=g) < q).squeeze(-1)
        grads = per_example_grads(params, X[idx], y[idx])
        norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in grads.values()))
        scale = (clip / (norms + 1e-12)).clamp(max=1.0)                     # shrink any gradient longer than clip
        for k in params:
            total = (grads[k] * scale.view(-1, *[1] * (grads[k].dim() - 1))).sum(0)
            if sigma > 0: total = total + torch.randn(total.shape, generator=g) * sigma * clip
            params[k] -= lr * total / (q * n)
    model.load_state_dict(params)
    return model

def mlp(d_in, n_out, hidden=32):
    return nn.Sequential(nn.Linear(d_in, hidden), nn.Tanh(), nn.Linear(hidden, n_out))

def accuracy(model, X, y):
    with torch.no_grad(): return (model(X).argmax(1) == y).float().mean().item()

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
_d = load_digits()
Xtr, Xte, ytr, yte = train_test_split(torch.tensor(_d.data / 16.0, dtype=torch.float32), torch.tensor(_d.target),
                                      test_size=0.25, random_state=0, stratify=_d.target)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

### 2. Check the accountant against a published number
Abadi et al. (2016) report epsilon about 1.26 for q = 0.01, sigma = 4, 10,000 steps and delta = 1e-5.

In [ ]:
e = epsilon(q=0.01, sigma=4.0, steps=10_000, delta=1e-5)
print("our accountant: epsilon = %.3f" % e)
assert 1.2 < e < 1.35

### 3. What clipping and noise do to one batch

In [ ]:
torch.manual_seed(0)
model = mlp(64, 10)
params = {k: v.detach() for k, v in model.named_parameters()}
one = lambda p, x, t: F.cross_entropy(functional_call(model, p, (x[None],)), t[None])
g = vmap(grad(one), in_dims=(None, 0, 0))(params, Xtr[:64], ytr[:64])
norms = torch.sqrt(sum((v.flatten(1) ** 2).sum(1) for v in g.values()))
print("per-example gradient norms: min %.2f  median %.2f  max %.2f" % (norms.min(), norms.median(), norms.max()))
print("share of examples clipped at clip = 1.0: %.2f" % (norms > 1.0).float().mean())

### 4. Accuracy against epsilon
Same steps and sampling rate each time. Only the noise changes.

In [ ]:
import matplotlib.pyplot as plt
STEPS, Q, DELTA = 300, 0.1, 1e-4                    # delta well below 1 / (number of training examples)
rows = []
for sigma in [8.0, 4.0, 2.0, 1.2, 0.8, 0.6]:
    torch.manual_seed(0)
    m = dp_sgd(mlp(64, 10), Xtr, ytr, sigma=sigma, clip=1.0, lr=1.0, steps=STEPS, q=Q)
    rows.append((sigma, epsilon(Q, sigma, STEPS, DELTA), accuracy(m, Xte, yte)))
    print("sigma %.1f   epsilon %7.2f   test accuracy %.3f" % rows[-1])
torch.manual_seed(0)
plain = dp_sgd(mlp(64, 10), Xtr, ytr, sigma=0.0, clip=float("inf"), lr=1.0, steps=STEPS, q=Q)
print("no privacy (no clipping, no noise)   test accuracy %.3f" % accuracy(plain, Xte, yte))
assert rows[-1][2] > rows[0][2] and accuracy(plain, Xte, yte) >= rows[0][2]
plt.semilogx([r[1] for r in rows], [r[2] for r in rows], "o-"); plt.xlabel("epsilon"); plt.ylabel("test accuracy"); plt.savefig("accuracy_vs_epsilon.png")

### 5. Three knobs that matter
- Larger batches (larger q) with more noise often beat small batches.
- The clipping length interacts with the learning rate: only their product matters once most gradients are clipped.
- Every run you try on the private data spends budget. Tune on public data when you can.

### 6. CIFAR-10 with Opacus (NOT RUN here)

In [ ]:
# NOT RUN: needs `pip install opacus`, torchvision, the CIFAR-10 download and a GPU.
import torchvision, torchvision.transforms as T
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator

tf = T.Compose([T.ToTensor(), T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])
train = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=tf)
loader = torch.utils.data.DataLoader(train, batch_size=1024, shuffle=True)
model = ModuleValidator.fix(torchvision.models.resnet18(num_classes=10))    # replaces BatchNorm, which DP-SGD cannot use
opt = torch.optim.SGD(model.parameters(), lr=0.5)
engine = PrivacyEngine()
model, opt, loader = engine.make_private_with_epsilon(module=model, optimizer=opt, data_loader=loader,
                                                      target_epsilon=8.0, target_delta=1e-5, epochs=20, max_grad_norm=1.0)
# ... the usual training loop ...
# print("epsilon spent:", engine.get_epsilon(1e-5))
# Repeat for target_epsilon in [1, 2, 4, 8] and plot test accuracy against epsilon.

### TODO
1. Keep epsilon near 3 and compare (q = 0.05, more steps) with (q = 0.2, fewer steps). Which wins?
2. Try clip = 0.1 and clip = 10 with the learning rate adjusted so that lr x clip stays fixed. What changes?
3. Run the Opacus cell. Does Opacus report a smaller epsilon than our accountant for the same settings? Why might it?